# Imprting Libraries

In [20]:
from transformers import AutoModelForCausalLM, T5Tokenizer, T5ForConditionalGeneration,DataCollatorForSeq2Seq,AutoModelForSeq2SeqLM
from transformers import AutoTokenizer, default_data_collator, get_linear_schedule_with_warmup,DataCollatorForLanguageModeling
from peft import get_peft_config, get_peft_model, PromptTuningInit, PromptTuningConfig, TaskType, PeftType
import torch
from datasets import load_dataset
import os
from torch.utils.data import DataLoader
from tqdm import tqdm


# LOADING MODEL

In [21]:
#model_name = 'bigscience/bloomz-560m'
#model_name = 'google/flan-t5-base'
model_name = 'gpt2-medium'
if model_name in ['gpt2','bigscience/bloomz-560m','gpt2-medium']:
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    #tokenizer.add_special_tokens({'pad_token': '[PAD]', 'sep_token': '[SEP]'})
    tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(model_name)
    model.resize_token_embeddings(len(tokenizer))
    
else:
    tokenizer = AutoTokenizer.from_pretrained(model_name,torch_dtype=torch.bfloat16)
    model =   AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Use a famous data

In [22]:
data = load_dataset("Abirate/english_quotes")

# Tokenize the quotes in the dataset using the specified tokenizer
data = data.map(lambda samples: tokenizer(samples["quote"]), batched=True)

Using the latest cached version of the dataset since Abirate/english_quotes couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'default' at C:\Users\utsav\.cache\huggingface\datasets\Abirate___english_quotes\default\0.0.0\7b544c4920a8be268b48b403c188acf0a462051b (last modified on Sun Jan 26 18:20:55 2025).


In [27]:
# new_sentences = "Be yourself; everyone else is already taken." 

# # Tokenize the new sentences
# inputs = tokenizer(new_sentences, return_tensors="pt", padding=True, truncation=True, max_length=64)
# print(inputs)

In [23]:
data["train"][0]

{'quote': '“Be yourself; everyone else is already taken.”',
 'author': 'Oscar Wilde',
 'tags': ['be-yourself',
  'gilbert-perreira',
  'honesty',
  'inspirational',
  'misattributed-oscar-wilde',
  'quote-investigator'],
 'input_ids': [447,
  250,
  3856,
  3511,
  26,
  2506,
  2073,
  318,
  1541,
  2077,
  13,
  447,
  251],
 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

# Inference without finetuining

In [24]:
new_sentences = "Be yourself" 

# Tokenize the new sentences
inputs = tokenizer(new_sentences, return_tensors="pt", padding=True, truncation=True, max_length=64)
print(inputs)
# Generate predictions
outputs = model.generate(inputs['input_ids'].to('cpu'), max_length=128)

# Decode the predictions
predictions = [tokenizer.decode(output, skip_special_tokens=True) for output in outputs]
print(predictions)

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


{'input_ids': tensor([[3856, 3511]]), 'attention_mask': tensor([[1, 1]])}
["Be yourself.\n\nYou're not alone.\n\nYou're not alone.\n\nYou're not alone.\n\nYou're not alone.\n\nYou're not alone.\n\nYou're not alone.\n\nYou're not alone.\n\nYou're not alone.\n\nYou're not alone.\n\nYou're not alone.\n\nYou're not alone.\n\nYou're not alone.\n\nYou're not alone.\n\nYou're not alone.\n\nYou're not alone.\n\nYou're not alone.\n\nYou're not alone.\n\nYou're not alone"]


In [35]:
import transformers
import datasets 
import peft
print(transformers.__version__)
print(datasets.__version__)
print(torch.__version__)
print(peft.__version__)

4.41.2
2.20.0
2.3.1+cpu
0.11.1


In [30]:
tokenized_train = data["train"].select(range(30))
tokenized_test =  data["train"].select(range(2))

In [31]:
tokenized_train[0]

{'quote': '“Be yourself; everyone else is already taken.”',
 'author': 'Oscar Wilde',
 'tags': ['be-yourself',
  'gilbert-perreira',
  'honesty',
  'inspirational',
  'misattributed-oscar-wilde',
  'quote-investigator'],
 'input_ids': [447,
  250,
  3856,
  3511,
  26,
  2506,
  2073,
  318,
  1541,
  2077,
  13,
  447,
  251],
 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [32]:
def print_number_of_trainable_model_parameters(model):
    trainable_model_params = 0
    all_model_params = 0
    for _, param in model.named_parameters():
        all_model_params += param.numel()
        if param.requires_grad:
            trainable_model_params += param.numel()
    return f'trainable model parameters: {trainable_model_params}\n \
            all model parameters: {all_model_params} \n \
            percentage of trainable model parameters: {(trainable_model_params / all_model_params) * 100} %'

# PEFT USING PROMPT

In [33]:
peft_config = PromptTuningConfig(
    task_type=TaskType.CAUSAL_LM,
    prompt_tuning_init=PromptTuningInit.RANDOM,
    num_virtual_tokens=3,
    #prompt_tuning_init_text="Complete the senetnce appropiately",
    tokenizer_name_or_path=model_name
)

peft_model = get_peft_model(model, peft_config)

print(print_number_of_trainable_model_parameters(peft_model))

if model_name in["gpt2",'bigscience/bloomz-560m',"gpt2-medium"]:
    #print(model_name)
    data_collator = DataCollatorForLanguageModeling(tokenizer,mlm=False)  
    #print()
else:
    data_collator = DataCollatorForSeq2Seq(tokenizer, model=peft_model)
    
print(model_name)
#print(data_collator)

trainable model parameters: 3072
             all model parameters: 354826240 
             percentage of trainable model parameters: 0.0008657758794839975 %
gpt2-medium


In [116]:
 #and 

In [36]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="sample-op-logs",
    #evaluation_strategy="no",
    no_cuda=True,  # This is necessary for CPU clusters.
    auto_find_batch_size=True,  # Find a suitable batch size that will fit into memory automatically
    learning_rate=3e-5,
    #per_device_train_batch_size=2,
    #per_device_eval_batch_size=2,
    #weight_decay=0.01,
    #evaluation_strategy="no",
    #do_eval=False,
    #save_total_limit=1,
    num_train_epochs=5    
    #predict_with_generate=True
)

from transformers import Trainer

trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=tokenized_train,
    #eval_dataset=tokenized_test,
    tokenizer=tokenizer,
    data_collator=data_collator
)

trainer.train()


In [36]:
trainer.save_model("SOFT-PROMPT")

In [37]:
tokenizer

BloomTokenizerFast(name_or_path='bigscience/bloomz-560m', vocab_size=250680, model_max_length=1000000000000000019884624838656, is_fast=True, padding_side='left', truncation_side='right', special_tokens={'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'pad_token': '</s>'}, clean_up_tokenization_spaces=False),  added_tokens_decoder={
	0: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("<s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("<pad>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}

# Inference with soft prompts 

In [23]:
#sample text with finetuning
from peft import PeftModel
new_sentences = "I am so clever that " 
#tokenizer = AutoTokenizer.from_pretrained(model_name)
# Tokenize the new sentences
inputs = tokenizer(new_sentences, return_tensors="pt", padding=True, truncation=True)
print(inputs)
loaded_model = PeftModel.from_pretrained(
    model,  # The base model to be used for prompt tuning
    "SOFT-PROMPT",   # The path where the trained Peft model is saved
    is_trainable=False  # Indicates that the loaded model should not be trainable
)

# Generate text using the loaded Peft model based on the provided input_ids and attention_mask.
loaded_model_outputs = loaded_model.generate(
    input_ids=inputs["input_ids"],
    attention_mask=inputs["attention_mask"],
    max_new_tokens=20,
    eos_token_id=tokenizer.eos_token_id
)

# Decode the generated token IDs into human-readable text.
decoded_output = tokenizer.batch_decode(loaded_model_outputs, skip_special_tokens=True)

# Print the decoded output, which represents the generated text.
print(decoded_output)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


{'input_ids': tensor([[   40,   716,   523, 14169,   326,   220]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1]])}


C:\Users\utsav\anaconda3\envs\mlep-w1-lab\lib\site-packages\peft\peft_model.py:1533: UserWarning: Position ids are not supported for parameter efficient tuning. Ignoring position ids.
  warnings.warn("Position ids are not supported for parameter efficient tuning. Ignoring position ids.")


['I am so clever that \xa0 the "I am a "I am a "I am a "I am a "I']


In [38]:
#sample text with finetuning
from peft import PeftModel
new_sentences = "I am so clever that " 
#tokenizer = AutoTokenizer.from_pretrained(model_name)
# Tokenize the new sentences
inputs = tokenizer(new_sentences, return_tensors="pt", padding=True, truncation=True)
print(inputs)
loaded_model = PeftModel.from_pretrained(
    model,  # The base model to be used for prompt tuning
    "SOFT-PROMPT",   # The path where the trained Peft model is saved
    is_trainable=False  # Indicates that the loaded model should not be trainable
)

# Generate text using the loaded Peft model based on the provided input_ids and attention_mask.
loaded_model_outputs = loaded_model.generate(
    input_ids=inputs["input_ids"],
    attention_mask=inputs["attention_mask"],
    max_new_tokens=20,
    eos_token_id=tokenizer.eos_token_id
)

# Decode the generated token IDs into human-readable text.
decoded_output = tokenizer.batch_decode(loaded_model_outputs, skip_special_tokens=True)

# Print the decoded output, which represents the generated text.
print(decoded_output)

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


{'input_ids': tensor([[    44,    912,   1427, 149014,    861,    210]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1]])}
['I am so clever that  I can not understand what you are saying']
